In [ ]:
r"""
Labeled concave partitions
==========================

See labeled_concave_partitions.ipynb for the annotated version.
"""

def Is_alm_decr(d):
    r"""Return 1 if the sequence ``d`` never increases by more than one."""
    l = len(d)
    for i in range(l - 1):
        if max(d[i + 1:]) - d[i] > 1:
            return 0
    return 1


def Is_concave(P):
    r"""Return 1 if the partition ``P`` is concave, 0 otherwise.

    Accepts any list-like; the empty partition is concave by convention.
    """
    P = list(P)
    l = len(P)
    if l == 0:
        return 1
    d = []
    for i in range(l - 1):
        d.append(P[i] - P[i + 1])
    d.append(P[l - 1])
    for k in range(1, l):
        dd = []
        for i in range(l - k + 1):
            dd.append(0)
            for j in range(k):
                dd[i] = dd[i] + d[i + j]
        if Is_alm_decr(dd) == 0:
            return 0
    return 1


def _strip_zeros(Q):
    r"""Drop trailing zeros, turning e.g. ``[2, 0]`` into a genuine partition ``[2]``."""
    Q = list(Q)
    while Q and Q[-1] == 0:
        Q.pop()
    return Q


def removable_cells(P):
    r"""
    The removable cells (external corners) of ``P``, ordered by row.

    A cell `(i, P_i - 1)` is removable when `P_i > P_{i+1}` (with `P_l = 0`).
    """
    P = list(P)
    l = len(P)
    return [(i, P[i] - 1) for i in range(l)
            if i == l - 1 or P[i] > P[i + 1]]


def addable_cells(P):
    r"""
    The addable cells (internal corners) of ``P``, ordered by row.

    A cell `(i, P_i)` is addable when `i = 0` or `P_{i-1} > P_i` (with `P_l = 0`).
    The empty partition has the single addable cell `(0, 0)`.
    """
    Q = list(P) + [0]
    return [(i, Q[i]) for i in range(len(Q))
            if i == 0 or Q[i - 1] > Q[i]]


def corner_neighbors(P):
    r"""
    Dict sending each addable cell of ``P`` to the tuple of removable cells adjacent to
    it along the boundary (0, 1 or 2 of them).
    """
    R = removable_cells(P)
    A = addable_cells(P)
    nb = {}
    for t, a in enumerate(A):
        s = []
        if t - 1 >= 0:
            s.append(R[t - 1])
        if t < len(R):
            s.append(R[t])
        nb[a] = tuple(s)
    return nb


def concavity_allowed_removable_cells(P):
    r"""
    The removable cells of ``P`` whose individual removal leaves a concave partition.

    This depends on ``P`` alone, not on which other cells happen to be labeled, so the
    pool of labelable removable cells is fixed once and for all and is independent of the
    order in which labels are added.
    """
    P = list(P)
    out = []
    for c in removable_cells(P):
        Q = list(P)
        Q[c[0]] -= 1
        if Is_concave(_strip_zeros(Q)):
            out.append(c)
    return out


class LabeledConcavePartition(SageObject):
    r"""
    A concave partition with a labeling of some of its corners.

    INPUT:

    - ``P`` -- a partition (any list-like of weakly decreasing positive integers);
      must be concave
    - ``labeled_removable`` -- iterable of removable cells `(i, j)` to label
    - ``labeled_addable`` -- iterable of addable cells `(i, j)` to label
    - ``check`` -- (default ``True``) verify concavity and the labeling rule
    """

    def __init__(self, P, labeled_removable=(), labeled_addable=(), check=True):
        self._partition = Partition(list(P))
        self._removable = tuple(removable_cells(self._partition))
        self._addable = tuple(addable_cells(self._partition))
        self._neighbors = corner_neighbors(self._partition)
        self._rlabels = frozenset(tuple(c) for c in labeled_removable)
        self._alabels = frozenset(tuple(c) for c in labeled_addable)
        self._allowed = None          # lazily filled cache
        if check:
            self._check()

    # -- validation --------------------------------------------------------

    def _check(self):
        if not Is_concave(self._partition):
            raise ValueError("the partition %s is not concave" % (self._partition,))
        bad = self._rlabels.difference(self._removable)
        if bad:
            raise ValueError("not removable cells of %s: %s"
                             % (self._partition, sorted(bad)))
        bad = self._alabels.difference(self._addable)
        if bad:
            raise ValueError("not addable cells of %s: %s"
                             % (self._partition, sorted(bad)))
        for a in self._alabels:
            missing = [r for r in self._neighbors[a] if r not in self._rlabels]
            if missing:
                raise ValueError(
                    "addable cell %s is labeled but its neighboring removable "
                    "cell(s) %s are not" % (a, missing))

    def is_valid(self):
        r"""Return ``True`` if this labeling satisfies the rule."""
        try:
            self._check()
        except ValueError:
            return False
        return True

    # -- accessors ---------------------------------------------------------

    def partition(self):
        return self._partition

    def removable_cells(self):
        return list(self._removable)

    def addable_cells(self):
        return list(self._addable)

    def allowed_removable_cells(self):
        r"""
        Removable cells that may ever be labeled: those whose removal keeps ``P``
        concave.  Computed once and cached.
        """
        if self._allowed is None:
            self._allowed = tuple(concavity_allowed_removable_cells(self._partition))
        return list(self._allowed)

    def neighbors(self, cell):
        r"""Removable cells adjacent to the addable cell ``cell``."""
        return list(self._neighbors[tuple(cell)])

    def labeled_removable(self):
        return sorted(self._rlabels)

    def labeled_addable(self):
        return sorted(self._alabels)

    def labeled_cells(self):
        return sorted(self._rlabels | self._alabels)

    def size(self):
        return self._partition.size()

    def number_of_labels(self):
        return len(self._rlabels) + len(self._alabels)

    # -- growing a labeling ------------------------------------------------

    def next_labels(self):
        r"""
        Cells that may be labeled next.

        An unlabeled addable cell qualifies when all of its neighbouring removable
        cells are already labeled; an unlabeled removable cell qualifies when removing
        that single cell from ``P`` leaves a concave partition.
        """
        next_cells = []
        for cell in self.allowed_removable_cells():
            if cell not in self._rlabels:
                next_cells.append(cell)
        for cell in self._addable:
            if cell not in self._alabels and \
                    all(r in self._rlabels for r in self._neighbors[cell]):
                next_cells.append(cell)
        return next_cells

    def label(self, cell):
        r"""Return a copy with ``cell`` labeled.  Raises if the result is invalid."""
        cell = tuple(cell)
        if cell in self._removable:
            return LabeledConcavePartition(self._partition,
                                           self._rlabels | {cell},
                                           self._alabels)
        if cell in self._addable:
            return LabeledConcavePartition(self._partition,
                                           self._rlabels,
                                           self._alabels | {cell})
        raise ValueError("%s is not a corner of %s" % (cell, self._partition))

    def unlabel(self, cell):
        r"""
        Return a copy with ``cell`` unlabeled.  Removing the label of a removable cell
        also removes the labels of the addable cells that depend on it.
        """
        cell = tuple(cell)
        R = set(self._rlabels)
        A = set(self._alabels)
        R.discard(cell)
        A.discard(cell)
        A = {a for a in A if all(r in R for r in self._neighbors[a])}
        return LabeledConcavePartition(self._partition, R, A)

    # -- enumeration -------------------------------------------------------

    @classmethod
    def labelings(cls, P, restricted=False):
        r"""
        All valid labelings of the concave partition ``P``.

        For every admissible subset `S` of removable cells, the addable cells available
        are exactly those all of whose neighbours lie in `S`; each such subset may be
        chosen freely.  With ``restricted=True`` only concavity-allowed removable cells
        are considered.
        """
        P = Partition(list(P))
        if not Is_concave(P):
            raise ValueError("the partition %s is not concave" % (P,))
        R = concavity_allowed_removable_cells(P) if restricted else removable_cells(P)
        nb = corner_neighbors(P)
        A = addable_cells(P)
        out = []
        for S in Subsets(R):
            S = frozenset(map(tuple, S))
            free = [a for a in A if all(r in S for r in nb[a])]
            for T in Subsets(free):
                out.append(cls(P, S, frozenset(map(tuple, T)), check=False))
        return out

    @classmethod
    def number_of_labelings(cls, P, restricted=False):
        r"""Number of valid labelings of ``P``, without building them."""
        P = Partition(list(P))
        R = concavity_allowed_removable_cells(P) if restricted else removable_cells(P)
        nb = corner_neighbors(P)
        A = addable_cells(P)
        total = 0
        for S in Subsets(R):
            S = frozenset(map(tuple, S))
            total += 2 ** sum(1 for a in A if all(r in S for r in nb[a]))
        return total

    # -- printing ----------------------------------------------------------

    def _repr_(self):
        return ("Concave partition %s with labeled removable %s, addable %s"
                % (list(self._partition),
                   set(self.labeled_removable()) or "{}",
                   set(self.labeled_addable()) or "{}"))

    def pp(self):
        r"""
        Pretty-print the Young diagram.

        ``.`` ordinary cell, ``r``/``R`` removable cell unlabeled/labeled,
        ``+``/``A`` addable cell unlabeled/labeled.
        """
        P = list(self._partition)
        width = (max(P) if P else 0) + 1
        rows = []
        for i in range(len(P) + 1):
            row = []
            for j in range(width):
                c = (i, j)
                if c in self._rlabels:
                    row.append('R')
                elif c in self._removable:
                    row.append('r')
                elif c in self._alabels:
                    row.append('A')
                elif c in self._addable:
                    row.append('+')
                elif i < len(P) and j < P[i]:
                    row.append('.')
                else:
                    row.append(' ')
            rows.append(' '.join(row).rstrip())
        print('\n'.join(rows))

    # -- comparison --------------------------------------------------------

    def __eq__(self, other):
        return (isinstance(other, LabeledConcavePartition)
                and self._partition == other._partition
                and self._rlabels == other._rlabels
                and self._alabels == other._alabels)

    def __ne__(self, other):
        return not self == other

    def __hash__(self):
        return hash((tuple(self._partition), self._rlabels, self._alabels))


def ConcavePartitions(n):
    r"""All concave partitions of ``n``."""
    return [P for P in Partitions(n) if Is_concave(P)]


def LabeledConcavePartitions(n, restricted=False):
    r"""All labeled concave partitions of size ``n``."""
    out = []
    for P in ConcavePartitions(n):
        out.extend(LabeledConcavePartition.labelings(P, restricted=restricted))
    return out


def labeled_concave_generating_series(N, restricted=False, q=None):
    r"""
    The series `\sum_{n \le N} \#\{\text{labeled concave partitions of } n\}\, q^n`.
    """
    if q is None:
        q = polygen(ZZ, 'q')
    return sum(len(LabeledConcavePartitions(n, restricted=restricted)) * q ** n
               for n in range(N + 1))